# 🎨 Udacity — Design of Computer Programs
### Peter Norvig


## 🃏 Lesson 1 — How to Write a Poker Program

### 🔍 Outlining the Problem

To write any program, follow 3 phases:
1. **Understand** — inventory the concepts of the problem
2. **Specify** — define it in terms that can become code
3. **Design** — structure it into code

---

### 📦 Concepts Inventory

| Concept | Description |
|---|---|
| **Card** | Has a `rank` (2–Ace) and a `suit` (♠ ♥ ♦ ♣) |
| **Hand** | A collection of 5 cards |
| **Hand Rank** | A value that determines how strong a hand is |
| **poker()** | Takes a list of hands → returns the best one |

---

### 🏆 Hand Rankings (strongest → weakest)

| # | Hand | Description |
|---|---|---|
| 1 | Straight Flush | 5 consecutive cards, same suit |
| 2 | Four of a Kind | 4 cards of the same rank |
| 3 | Full House | 3 of a kind + a pair |
| 4 | Flush | 5 cards of the same suit |
| 5 | Straight | 5 consecutive cards, any suit |
| 6 | Three of a Kind | 3 cards of the same rank |
| 7 | Two Pair | 2 different pairs |
| 8 | One Pair | 2 cards of the same rank |
| 9 | High Card | None of the above |

---

### 🧩 Key Concepts for Hand Rank Logic
- **n-of-a-kind** → n cards of the same rank (e.g. two 2s, two Jacks)
- **Straight** → 5 consecutive ranks (e.g. 5-6-7-8-9)
- **Flush** → 5 cards of the same suit (e.g. all ♠)
- **Ties** → when two hands have the same rank, compare by **card values** (higher card wins)

---

### 🎯 Goal of the `poker()` function

```python
def poker(hands):
    """Takes a list of hands, returns the best hand."""
    return max(hands, key=hand_rank)
```

> 💡 Using `max()` with a `key` function is a clean **functional programming** pattern — no loops needed.


### 🗂️ Representing Hands — Choosing a Data Structure

**Best choice → list of tuples** `[(rank, suit), ...]` ✅

| Option | Example | Why not? |
|---|---|---|
| List of strings | `['JS', 'JD', 'JC']` | Hard to extract rank/suit — need to parse the string |
| **List of tuples** ✅ | `[(11,'S'), (11,'D'), (11,'C')]` | Rank and suit already separated → easy to work with |
| Set of tuples | `{(11,'S'), (11,'D'), ...}` | Unordered — can't rely on card position |

```python
card = (11, 'S')   # Jack of Spades
card[0]            # → 11  (rank)
card[1]            # → 'S' (suit)
```


### 🎯 Poker Function & Testing

**Goal:** `poker(hands)` takes a list of hands and returns the best one.

```python
def poker(hands):
    "Return the best hand: poker([hand,...]) => hand"
    return max(hands, key=hand_rank)

def hand_rank(hand):
    return None  # placeholder — to be implemented later
```

> 💡 `max()` with `key=hand_rank` picks the hand with the highest rank value. Clean and no loops needed.

---

### ✅ Testing with `assert`

Write a `test()` function with **assert statements** — if an assertion is False, Python stops and raises an error immediately.

```python
def test():
    sf = "6C 7C 8C 9C TC".split()  # Straight Flush
    fk = "9D 9H 9S 9C 7D".split()  # Four of a Kind
    fh = "TD TC TH 7C 7D".split()  # Full House

    assert poker([sf, fk, fh]) == sf   # straight flush beats all
    assert poker([fk, fh]) == fk       # four of a kind beats full house
    assert poker([fh, fh]) == fh       # tie → returns first fh
```

**Why these test cases?**
- Test the **happy path** (sf wins)
- Test a **specific matchup** (fk vs fh)
- Test a **tie** — `max()` returns the first element when values are equal


### 🏅 hand_rank() — Returning a Comparable Value

**Goal:** `hand_rank(hand)` must return a value that allows comparing two hands correctly.

**Problem with returning just a single integer (0–8):**
```python
pair_of_10s → 1
pair_of_9s  → 1   # same rank! but 10s should win
```
→ A single number is **not enough** to break ties within the same rank.

---

### 🔢 Three Options to Fix This

| Option | Four-of-9s-with-5 example | Works? | Best? |
|---|---|---|---|
| Large integer | `70905` | ✅ | ❌ complicated arithmetic |
| Float | `7.0905` | ✅ | ❌ precision issues |
| **Tuple** | `(7, 9, 5)` | ✅ | ✅ simple, clean |

---

### ✅ Why Tuples Win

No arithmetic needed — just use commas:
```python
hand_rank(fk_9s) → (7, 9, 5)
#                   ↑  ↑  ↑
#                   │  │  └── kicker (remaining card)
#                   │  └───── rank of the 4-of-a-kind (9s)
#                   └──────── hand type: Four of a Kind = 7
```

---

### 📚 Lexicographic Ordering

Python compares tuples (and strings) **left to right** — this is called **lexicographic ordering**:

```python
(7, 9, 5) > (7, 3, 2)
# 7 == 7 → tie, move to next
# 9 > 3  → (7, 9, 5) wins ✅

'help' > 'hello'
# h==h, e==e, l==l → move on
# p > l → 'help' wins ✅
```

> 💡 Same principle used for sorting words in a dictionary — that's why it's called **lexico**graphic (lex = words).


### 🃏 hand_rank() — Full Tuple Structure per Hand

Each hand is described as a tuple: `(hand_type, tiebreakers...)`

| # | Hand | Example | Tuple returned |
|---|---|---|---|
| 8 | Straight Flush | J high | `(8, 11)` |
| 7 | Four of a Kind | Four Aces + Q kicker | `(7, 14, 12)` |
| 6 | Full House | Eights over Kings | `(6, 8, 13)` |
| 5 | Flush | 10, 8 high | `(5, 10, 8, ...)` all 5 cards |
| 4 | Straight | J high | `(4, 11)` |
| 3 | Three of a Kind | Three 7s | `(3, 7, 5, 2)` |
| 2 | Two Pair | Jacks and 3s | `(2, 11, 3, kicker)` |
| 1 | One Pair | Pair of 2s, J high | `(1, 2, 11, ...)` all cards |
| 0 | High Card | Nothing | `(0, ranks...)` all cards high→low |

**Key rules:**
- The **first number** = hand type (0–8)
- The **remaining numbers** = tiebreakers in order of importance
- For ties within the same hand type, Python compares the tuple **left to right** (lexicographic order)
- Cards use numeric ranks: `Jack=11, Queen=12, King=13, Ace=14`


### 🔧 Implementing hand_rank() with Tuples

`hand_rank()` returns a tuple where:
- **Element 0** → hand type (0–8)
- **Remaining elements** → tiebreakers in order of importance

```python
# Straight flush examples:
# ranks 2-3-4-5-6  → (8, 6)   ← high card is 6
# ranks 6-7-8-9-T  → (8, 10)  ← high card is 10
# (8, 10) > (8, 6) → second hand wins ✅
```

---

### 🧩 The `kind(n, ranks)` Function

`kind(n, ranks)` checks if there are **n cards of the same rank** and returns that rank (or `None`/falsy if not found).

```python
kind(4, ranks)  # → 9  if hand has four 9s  (truthy ✅)
kind(4, ranks)  # → None if no four of a kind (falsy ❌)
```

**Dual purpose — used as both:**
- A **value** → returns the rank (e.g. `9` for four 9s)
- A **boolean test** → truthy if found, falsy if not

```python
# In hand_rank():
if kind(4, ranks):              # ← boolean test
    return (7, kind(4, ranks),  # ← value usage
               kind(1, ranks))
```

> ⚠️ This works because card ranks go from **2 to 14** — `0` is never a valid rank, so returning `0` or `None` as a "not found" value is safe and always falsy in Python.

> 📝 **Note:** Unlike Java, Python treats `0`, `None`, and empty values as falsy — this dual-return pattern is a clean Python idiom.


### ✅ Testing hand_rank() — Adding Assertions per Hand

```python
sf = "6C 7C 8C 9C TC".split()  # Straight Flush  → high card = T (10)
fk = "9D 9H 9S 9C 7D".split()  # Four of a Kind  → four 9s, kicker 7
fh = "TD TC TH 7C 7D".split()  # Full House       → three 10s, pair of 7s

assert hand_rank(sf) == (8, 10)    # straight flush, 10 high
assert hand_rank(fk) == (7, 9, 7)  # four of a kind: four 9s + 7 kicker
assert hand_rank(fh) == (6, 10, 7) # full house: three 10s + pair of 7s
```

**How to read each tuple:**

| Hand | Tuple | Explanation |
|---|---|---|
| `sf` (6C 7C 8C 9C TC) | `(8, 10)` | Straight flush = 8, high card = T = 10 |
| `fk` (9D 9H 9S 9C 7D) | `(7, 9, 7)` | Four of a kind = 7, four 9s, kicker = 7D |
| `fh` (TD TC TH 7C 7D) | `(6, 10, 7)` | Full house = 6, three 10s, pair of 7s |

> 💡 Notice `fk` returns `(7, 9, 7)` — the first `7` is the **hand type** (four of a kind), and the second `7` is the **kicker card** (7 of diamonds). Two different 7s meaning different things!


### 🔧 hand_rank() — Complete Solution (All 9 Hand Types)

**Exercise:** fill in the return tuples for hands 6 → 0 using `kind()`, `flush()`, `straight()`, `two_pair()`, and `card_ranks()`.

```python
def hand_rank(hand):
    ranks = card_ranks(hand)
    if straight(ranks) and flush(hand):              # 8 — straight flush
        return (8, max(ranks))
    elif kind(4, ranks):                             # 7 — four of a kind
        return (7, kind(4, ranks), kind(1, ranks))
    elif kind(3, ranks) and kind(2, ranks):          # 6 — full house
        return (6, kind(3, ranks), kind(2, ranks))
    elif flush(hand):                                # 5 — flush
        return (5, ranks)
    elif straight(ranks):                            # 4 — straight
        return (4, max(ranks))
    elif kind(3, ranks):                             # 3 — three of a kind
        return (3, kind(3, ranks), ranks)
    elif two_pair(ranks):                            # 2 — two pair
        return (2, two_pair(ranks), ranks)
    elif kind(2, ranks):                             # 1 — one pair
        return (1, kind(2, ranks), ranks)
    else:                                            # 0 — high card
        return (0, ranks)
```

**Design decisions explained:**

| Hand | Tuple structure | Tiebreakers |
|---|---|---|
| Straight flush (8) | `(8, max(ranks))` | highest card |
| Four of a kind (7) | `(7, quad_rank, kicker)` | rank of the 4, then kicker |
| Full house (6) | `(6, triple_rank, pair_rank)` | rank of 3-of-a-kind first, then pair |
| Flush (5) | `(5, ranks)` | all 5 ranks in descending order |
| Straight (4) | `(4, max(ranks))` | highest card |
| Three of a kind (3) | `(3, triple_rank, ranks)` | rank of triple + remaining descending |
| Two pair (2) | `(2, (high_pair, low_pair), ranks)` | both pairs + kicker |
| One pair (1) | `(1, pair_rank, ranks)` | pair rank + remaining descending |
| High card (0) | `(0, ranks)` | all 5 ranks descending |

> 💡 `card_ranks()` returns ranks **sorted highest → lowest**, so when there is no special hand, just returning the full ranks list is enough — the highest card is automatically the first tiebreaker.

> 💡 For flush and high card, the whole `ranks` tuple is the tiebreaker — Python compares tuples element by element, so this correctly handles all edge cases automatically.


### 🃏 Helper Function: card_ranks()

**Purpose:** convert a hand (list of card strings) into a list of ranks **sorted highest → lowest**.

> 💡 Always sorted descending because ranks are used as tiebreakers — the highest card must come first.

**New test assertions added to `test()`:**

```python
assert card_ranks(sf) == [10, 9, 8, 7, 6]   # straight flush: T=10, descending
assert card_ranks(fk) == [9, 9, 9, 9, 7]    # four 9s first, then kicker 7
assert card_ranks(fh) == [10, 10, 10, 7, 7]  # three 10s first, then pair of 7s
```

| Hand | Cards | card_ranks output | Why? |
|---|---|---|---|
| `sf` | 6C 7C 8C 9C TC | `[10, 9, 8, 7, 6]` | T=10, sorted high→low |
| `fk` | 9D 9H 9S 9C 7D | `[9, 9, 9, 9, 7]` | four 9s before the 7 kicker |
| `fh` | TD TC TH 7C 7D | `[10, 10, 10, 7, 7]` | three 10s before the pair of 7s |

> 💡 `card_ranks()` only cares about **rank**, not suit. Face cards map to integers: J=11, Q=12, K=13, A=14, T=10.


### 🔧 Implementing card_ranks() — Mapping Face Cards to Integers

**Problem:** pulling the rank character directly gives wrong ordering — `'T'` sorts alphabetically *after* `'A'`, `'Q'`, etc., which is incorrect.

**Solution:** use `.index()` on a reference string where position = value:

```python
def card_ranks(cards):
    "Return a list of the ranks, sorted with higher first."
    ranks = ['--23456789TJQKA'.index(r) for r, s in cards]
    ranks.sort(reverse=True)
    return ranks

# card_ranks(['AC', '3D', '4S', 'KH']) => [14, 13, 4, 3]
```

**How the index trick works:**

```
'--23456789TJQKA'
  0123456789...
         ↑ T is at index 10 → maps to 10
              ↑ J = 11, Q = 12, K = 13, A = 14
```

| Character | Index in string | Numeric rank |
|---|---|---|
| `'2'` | 2 | 2 |
| `'9'` | 9 | 9 |
| `'T'` | 10 | 10 |
| `'J'` | 11 | 11 |
| `'Q'` | 12 | 12 |
| `'K'` | 13 | 13 |
| `'A'` | 14 | 14 |

> 💡 The `--` at position 0 and 1 are placeholder dashes — no card has rank 0 or 1. This keeps the indexing aligned so `'2'` lands at index 2 naturally.

> 💡 Alternative approaches (dict lookup, if-elif chain) would work too, but the index string is the most concise.


### 🔀 Helper Functions: straight() and flush()

**Tests added first:**

```python
assert straight([9, 8, 7, 6, 5]) == True   # consecutive ranks → straight
assert straight([9, 8, 8, 6, 5]) == False  # pair: not a straight
assert flush(sf) == True                    # sf = all clubs → flush
assert flush(fk) == False                   # fk = 4 different suits → not a flush
```

**Solutions:**

```python
def straight(ranks):
    "Return True if the ordered ranks form a 5-card straight."
    return (max(ranks) - min(ranks) == 4) and (len(set(ranks)) == 5)

def flush(hand):
    "Return True if all the cards have the same suit."
    suits = [s for r, s in hand]
    return len(set(suits)) == 1
```

**straight() logic:**

| Condition | Why? |
|---|---|
| `max - min == 4` | 5 consecutive cards span exactly 4 (e.g. 5→9: 9−5=4) |
| `len(set(ranks)) == 5` | all 5 ranks are different (rules out pairs like [9,8,8,6,5]) |

> 💡 Both conditions together are enough — 5 distinct ranks spanning 4 = guaranteed consecutive sequence.

**flush() logic:**

- Extract suits: `[s for r, s in hand]` — each card is a 2-char string, `s` is the second character
- `set(suits)` collapses duplicates → if only 1 unique suit remains, all cards share the same suit


### 🔧 Helper Function: kind(n, ranks)

**Goal:** return the first rank that appears exactly `n` times in the hand. Return `None` if not found.

```python
def kind(n, ranks):
    """Return the first rank that this hand has exactly n of.
    Return None if there is no n-of-a-kind in the hand."""
    for r in ranks:
        if ranks.count(r) == n:
            return r
    return None
```

**How it works:**
- Loop through each rank in the (sorted) list
- `.count(r)` counts how many times `r` appears in the list
- If it matches exactly `n` → return that rank
- If nothing matches → return `None` (falsy)

---

**Test assertions:**

```python
tp = "5S 5D 9H 9C 6S".split()  # Two Pair: pair of 9s and pair of 5s, kicker 6
fkranks = card_ranks(fk)        # [9, 9, 9, 9, 7]
tpranks = card_ranks(tp)        # [9, 9, 6, 5, 5]

assert kind(4, fkranks) == 9    # four 9s → returns 9
assert kind(3, fkranks) == None # no three of a kind (it's four!) → None
assert kind(2, fkranks) == None # no pair → None
assert kind(1, fkranks) == 7    # one 7 (the kicker) → returns 7
```

**Key insight — exactly n, not at least n:**

| Call | fkranks = [9,9,9,9,7] | Why? |
|---|---|---|
| `kind(4, fkranks)` | `9` | four 9s → exactly 4 ✅ |
| `kind(3, fkranks)` | `None` | 9 appears 4 times, not 3 ❌ |
| `kind(2, fkranks)` | `None` | no rank appears exactly 2 times ❌ |
| `kind(1, fkranks)` | `7` | 7 appears exactly once ✅ |

> 💡 `kind()` uses `.count()` — a built-in list method that counts occurrences of an element.
> Since ranks are sorted **highest → lowest**, it always returns the **highest** matching rank first.


### 🔧 Helper Function: two_pair(ranks)

**Goal:** if there are two pairs, return `(highest_pair, lowest_pair)`; otherwise `None`.

```python
def two_pair(ranks):
    """If there are two pair, return the two ranks as a
    tuple: (highest, lowest); otherwise return None."""
    pair  = kind(2, ranks)           # highest pair (ranks sorted high→low)
    lowpair = kind(2, ranks[::-1])   # lowest pair (scan from the end)
    if pair and lowpair and pair != lowpair:
        return (pair, lowpair)
    return None
```

**How it works:**

| Step | Code | Result for `tpranks = [10, 10, 9, 7, 7]`* |
|---|---|---|
| Find highest pair | `kind(2, ranks)` | `10` — first pair found left→right |
| Find lowest pair | `kind(2, ranks[::-1])` | `7` — first pair found right→left |
| Check they differ | `pair != lowpair` | `10 != 7` ✅ → return `(10, 7)` |
| Single pair case | `pair == lowpair` | same rank = only 1 pair → `None` |

> 💡 `ranks[::-1]` reverses the list — since ranks are sorted **high → low**, reversing gives **low → high**, so `kind()` finds the lowest pair first.

> 💡 The check `pair != lowpair` is necessary to avoid returning `(9, 9)` when there's a three-of-a-kind — `kind(2, ...)` would find the same rank from both ends.

---

### ✅ Lesson 1 — Complete Program Summary

All helper functions implemented and tested:

```python
def card_ranks(hand):   # rank chars → sorted ints [14..2]
def straight(ranks):    # max-min==4 and 5 unique ranks
def flush(hand):        # all suits identical
def kind(n, ranks):     # first rank appearing exactly n times
def two_pair(ranks):    # (high_pair, low_pair) or None
def hand_rank(hand):    # returns comparable tuple (0–8, tiebreakers...)
def poker(hands):       # max(hands, key=hand_rank)
```

> 💡 Norvig closes Lesson 1 noting the test suite is not exhaustive yet — in the next videos he'll show an edge case that breaks the program and introduce more rigorous testing strategies.


---

## 🐛 Edge Case: Ace-Low Straight (A-2-3-4-5)

The Ace can count as **low** only in `A-2-3-4-5` (the "wheel"). Our program breaks this in 2 ways:
- `ranks = [14,5,4,3,2]` → max-min = 12 ≠ 4 → **not detected as straight**
- Even if detected, `max(ranks) = 14` → **wrong high card** (should be 5)

**Naïve fix:** change `card_ranks()`, `straight()`, and `hand_rank()`.

### 💡 Norvig's Design Principle
> *"The amount of change in code should be proportional to the amount of change in the concept."*

One conceptual change → one code change. Fix it **only in `card_ranks()`**: replace `[14,5,4,3,2]` with `[5,4,3,2,1]`. The other functions need zero changes.


### ✅ Fix: Ace-Low Straight in card_ranks()

Only `card_ranks()` needs to change — one special-case check at the end:

```python
def card_ranks(hand):
    "Return a list of the ranks, sorted with higher first."
    ranks = ['--23456789TJQKA'.index(r) for r, s in hand]
    ranks.sort(reverse=True)
    return [5, 4, 3, 2, 1] if ranks == [14, 5, 4, 3, 2] else ranks
```

**Why this works:**
- `[14, 5, 4, 3, 2]` is the only case where Ace plays low
- Replaced with `[5, 4, 3, 2, 1]` → `max-min = 5-1 = 4` ✅ and 5 unique values ✅
- `straight()` and `hand_rank()` need **zero changes**

```python
assert straight(card_ranks("AC 2D 4H 3D 5S".split())) == True  ✅
```

> 💡 `1` is only ever returned in this single edge case — all other ranks are 2–14.


### 🔁 Handling ties with `allmax`

Define a helper that returns all items equal to the maximum (optionally using a key).

```python
def allmax(iterable, key=None):
    result = []
    best = None
    for x in iterable:
        k = x if key is None else key(x)
        if not result or k > best:
            result = [x]
            best = k
        elif k == best:
            result.append(x)
    return result

def poker(hands):
    """Return list of winning hand(s)."""
    return allmax(hands, key=hand_rank)
```

Tests below ensure ties are handled (see next cell).

### 🎴 Dealing Cards – the `deal` function

You need a helper that takes:

```python
def deal(numhands, n=5, deck=mydeck):
    """Return a list of `numhands` poker hands with `n` cards each.
    `deck` is a list of card strings (default 52-card deck).
    """
```

**What to do:**
1. **Make a copy of the deck** so the caller's deck isn't mutated:
   `d = deck[:]`
2. **Shuffle** the copy using `random.shuffle(d)` (in-place modification).
3. **Slice out** `n` cards for each hand. With hands numbered 0..numhands‑1,
   the `i`th hand is `d[i*n:(i+1)*n]`.
4. **Return** the list of all hands.

```python
import random

mydeck = [r+s for r in '23456789TJQKA' for s in 'SHDC']

def deal(numhands, n=5, deck=mydeck):
    d = deck[:]                # copy
    random.shuffle(d)          # random order
    return [d[i*n:(i+1)*n] for i in range(numhands)]
```

> 💡 using a default deck lets you call `deal(4)` to get four 5‑card hands.
> You can also pass a custom deck or change `n` to deal 7‑card hands.

The quiz text hints at `random.shuffle`; the list comprehension forms the deck itself (all combinations of ranks and suits).

**Example usage:**

```python
>>> deal(2)               # two 5-card hands
[['7H', 'KD', '2C', 'AS', '9D'],
 ['5S', '3H', 'TC', 'AH', '4D']]

>>> deal(3, n=7)          # three 7-card hands
[['QC','2S','9H','8D','5C','JD','6S'],
 ['3C','KH','4S','6D','TS','AD','7H'],
 ['2D','JC','8S','KD','9C','4H','QS']]
```

Each sublist is one hand; the order is random so you get different cards every time. You can see the slicing: the first 5 (or 7) cards go to player 1, the next 5 (or 7) to player 2, etc.

## 📊 Poker Hand Probabilities – Validating Our Code

### The Wikipedia Reference Table

Wikipedia lists the probability of each poker hand type out of all possible 5-card hands:

| Rank | Hand | Approximate % |
|------|------|--------------|
| 1 (best) | Straight Flush | ~0.0015% |
| 2 | Four of a Kind | ~0.024% |
| 3 | Full House | ~0.14% |
| 4 | Flush | ~0.20% |
| 5 | Straight | ~0.39% |
| 6 | Three of a Kind | ~2.11% |
| 7 | Two Pair | ~4.75% |
| 8 | One Pair | ~42.26% |
| 9 (worst) | High Card | ~50.12% |

> Key insight: **rarer hands rank higher** — poker hand rankings are perfectly ordered by probability. The game is mathematically sound.

---

### How Many Hands Do We Need to Sample?

To validate our `hand_rank` function against these probabilities we need to simulate many random deals. But how many?

| Option | Verdict |
|--------|---------|
| ~52 (one per card) | ❌ Far too few — number of cards is irrelevant |
| ~1,000 per card (~52,000) | ❌ Still not enough for the rarest hands |
| **~700,000** | ✅ Target answer |
| 52! (all permutations) | ❌ Astronomically large, unnecessary |

**The reasoning:**  
We want **at least ~10 occurrences of the least common hand** (straight flush, ≈ 1 in 65,000) so that random variance doesn't skew our estimate badly.

$$\text{needed samples} \approx \frac{10}{P(\text{straight flush})} \approx \frac{10}{0.0000154} \approx 700{,}000$$

If we only expected 1 straight flush, sampling noise would make results meaningless (0 or 2 are equally likely). With ~10, we're reliably in the right ballpark.

---

### `hand_percentages` – Simulating the Table

```python
import random

def hand_percentages(n=700_000):
    """
    Deal n random 5-card hands, count how many of each rank appear,
    and print the percentage for each hand type.
    Default n=700,000 (takes ~20-30 sec; use n=1000 for a quick test).
    """
    counts = [0] * 9          # one bucket per rank (index 0 = straight flush … 8 = high card)
    for _ in range(n // 10):  # deal 10 hands at a time for efficiency
        for hand in deal(10):
            counts[hand_rank(hand)[0]] += 1
    for i in range(9):
        print(f"{hand_names[i]:20s}: {100 * counts[i] / n:5.2f}%")
```

> ⚠️ Running with 700,000 deals takes 20-30 seconds. Use `n=1_000` for a quick check.

---

### Comparing Simulated vs. Exact Results

When run with a large `n`, the output looks something like:

```
straight_flush      :  0.00%   (Wikipedia: 0.0015%)
four_of_a_kind      :  0.02%   (Wikipedia: 0.024%)
full_house          :  0.14%   (Wikipedia: 0.144%)
flush               :  0.20%   (Wikipedia: 0.197%)
straight            :  0.38%   (Wikipedia: 0.392%)
three_of_a_kind     :  2.09%   (Wikipedia: 2.113%)
two_pair            :  4.72%   (Wikipedia: 4.754%)
pair                : 42.30%   (Wikipedia: 42.257%)
high_card           : 50.15%   (Wikipedia: 50.118%)
```

The simulated percentages closely match the exact mathematical values — confirming our `hand_rank` logic is correct. This is a form of **empirical verification via Monte Carlo simulation**.


## 🧭 Dimensions of Programming

A program lives in a **multidimensional space** defined by four key axes. Every design decision moves it along one or more of these dimensions:

| Axis | Question |
|------|----------|
| ✅ **Correctness** | Does it do the right thing? |
| ⚡ **Efficiency** | Is it fast enough? |
| 🧩 **Features** | What exactly does it do? |
| 💎 **Elegance** | Is it clear, simple, and general? |

```
        Correctness
             ▲
             │         ★ ideal program
             │        /
             │       /
─────────────┼──────────────► Features
             │      /
             │     /
         (Efficiency & Elegance are the other axes — think 4D space)
```

### 💎 Elegance is not optional
> *"Elegance is not optional."* — Richard O'Keefe

Elegance is made up of **"-ity" qualities**:
- **Clarity** — easy to read and understand
- **Simplicity** — no unnecessary complexity
- **Generality** — works beyond just the current case

Improving elegance doesn't add features today, but it **pays off in the future**: a more elegant program is cheaper to maintain, extend, and debug.

### ⚖️ Voltaire's Rule
> *"The best is the enemy of the good."*

Don't chase 100% perfection in one dimension at the cost of all others. Good engineering means making **smart tradeoffs** — knowing when to stop and move on.


## ♻️ Refactoring – Moving Along the Elegance Axis

**Refactoring** = changing a program to be cleaner/clearer *without* changing what it does.

The original `hand_rank` repeated itself checking for pairs and triples (`3 of a kind AND 2 of a kind` → full house). That violates the **DRY principle**:

> *"Don't Repeat Yourself"* — every piece of knowledge should appear once and only once.

---

### The `group` Helper

Instead of asking "is there a pair? is there a triple?" separately, we create a richer representation upfront:

```python
def group(hand):
    """Return (counts, ranks) sorted by count desc, then rank desc."""
    ranks = [r for r, s in hand]
    return (sorted([ranks.count(r) for r in set(ranks)], reverse=True),
            sorted(set(ranks), key=ranks.count, reverse=True))
```

**Example** — hand `[7, 10, 7, 9, 7]`:

| | Value |
|---|---|
| `counts` | `[3, 1, 1]` — three 7s, one 10, one 9 |
| `ranks`  | `[7, 10, 9]` — ordered by count |

---

### Refactored `hand_rank`

```python
def hand_rank(hand):
    groups = group(hand)
    counts, ranks = groups
    if ranks == (14, 5, 4, 3, 2):   # ace-low straight
        ranks = (5, 4, 3, 2, 1)
    straight = len(set(ranks)) == 5 and max(ranks) - min(ranks) == 4
    flush    = len(set(s for r, s in hand)) == 1
    return (9 if counts == [5]          else   # five of a kind
            8 if straight and flush     else   # straight flush
            7 if counts == [4, 1]       else   # four of a kind
            6 if counts == [3, 2]       else   # full house
            5 if flush                  else   # flush
            4 if straight               else   # straight
            3 if counts == [3,1,1]      else   # three of a kind
            2 if counts == [2,2,1]      else   # two pair
            1 if counts == [2,1,1,1]    else   # one pair
            0), ranks                          # high card
```

No repetition — each hand type is checked **once** using its count signature.

---

### 🤯 Bonus Insight – Poker Hands as Partitions of 5

The count patterns are exactly the **integer partitions of 5**:

| Partition | Hand |
|-----------|------|
| `[5]` | Five of a kind |
| `[4,1]` | Four of a kind |
| `[3,2]` | Full house |
| `[3,1,1]` | Three of a kind |
| `[2,2,1]` | Two pair |
| `[2,1,1,1]` | One pair |
| `[1,1,1,1,1]` | High card |

All **7 integer partitions of 5**, in **lexicographic order** — which is also exactly the **poker ranking order** from best to worst. Pure math, built into the game.


## 🔧 `group`, `unzip`, and the Lookup Table Approach

### `group` and `unzip`

```python
def group(items):
    """Return a sorted list of (count, item) pairs, highest count first."""
    return sorted([(items.count(x), x) for x in set(items)], reverse=True)

def unzip(pairs):
    """Convert a list of (count, item) pairs into (counts, items)."""
    return zip(*pairs)
```

- `group` counts occurrences of each item and sorts biggest-first.
- `unzip` (via `zip(*pairs)`) converts a list of pairs into a pair of lists — a standard Python trick.

---

### Lookup Table – One More Refactor

Instead of a long `if/elif` chain, store the partition → ranking mapping in a dict:

```python
count_rankings = {(5,): 10, (4,1): 7, (3,2): 6,
                  (3,1,1): 3, (2,2,1): 2, (2,1,1,1): 1, (1,1,1,1,1): 0}

def hand_rank(hand):
    counts, ranks = unzip(group(hand))
    if ranks == (14,5,4,3,2): ranks = (5,4,3,2,1)
    straight = len(set(ranks)) == 5 and max(ranks) - min(ranks) == 4
    flush    = len(set(s for r,s in hand)) == 1
    return max(count_rankings[counts], 4*straight + 5*flush), ranks
```

The key trick: `4*straight + 5*flush` exploits bool→int conversion:

| straight | flush | expression | result |
|----------|-------|-----------|--------|
| False | False | 0+0 | 0 |
| True | False | 4+0 | 4 ✅ straight |
| False | True | 0+5 | 5 ✅ flush |
| True | True | 4+5 | 9 ✅ straight flush |

9 lines of `if/elif` collapsed to **1 line** — more concise, but less explicit. Both are valid; it's a matter of taste.

---

## 📝 Lessons Learned

| Step | Principle |
|------|-----------|
| 1 | **Understand the problem** — read the spec, ask questions |
| 2 | **Define the pieces** — cards, hands, ranks, suits… |
| 3 | **Reuse what exists** — `max()`, `random.shuffle()`, `zip()` |
| 4 | **Write tests** — you don't know what your program does without them |
| 5 | **Explore the design space** — correctness → efficiency → features → elegance |
| 6 | **Use good taste to know when to stop** — don't over-engineer |


---

# 🃏 Bonus: Shuffling

## The Teacher's Shuffle vs. Knuth's Algorithm P

### Teacher's shuffle (❌ wrong)
```python
# swap random pairs until every position has been swapped at least once
swapped = [False] * N
while not all(swapped):
    i, j = random_indices()
    swap(deck[i], deck[j])
    swapped[i] = swapped[j] = True
```
**Problems:**
- **Not guaranteed to terminate** — the while-loop could theoretically run forever.
- **O(N²)** — the last unswapped element takes ~N tries to hit, so total ≈ N×N swaps.
- **Biased** — not all permutations are equally probable.

---

### Knuth's Algorithm P (✅ correct)
```python
def shuffle(deck):
    N = len(deck)
    for i in range(N - 1):
        j = random.randrange(i, N)   # pick from i..N-1
        deck[i], deck[j] = deck[j], deck[i]
```
- **O(N)** — exactly one swap per element.
- **Fair** — every permutation is equally probable.
- Using `N` instead of `N-1` in the range would be fine (still fair) but wastes one no-op swap at the end.

---

## Comparing Shuffles

| Algorithm | Complexity | Fair? |
|-----------|-----------|-------|
| `random.shuffle` (Knuth) | O(N) | ✅ |
| Teacher's `shuffle1` | O(N²) | ❌ biased |
| `shuffle2` (partial fix) | O(N²) | ✅ correct but slow |
| `shuffle3` (swap each with any) | O(N) | ❌ biased |

> **Use `random.shuffle`.** O(N) and provably fair.

---

## Pure Functions vs. Subroutines

| | Pure function | Subroutine (impure) |
|---|---|---|
| Example | `sorted(lst)` | `lst.sort()` |
| Returns | new value | `None` |
| Modifies input? | ❌ | ✅ |
| Test in 1 line? | ✅ `assert sorted([3,2,1]) == [1,2,3]` | ❌ need setup + call + inspect |

**Prefer pure functions** when possible — they are easier to test and reason about. Use subroutines only when managing significant shared state.


---

# 🐍 Andy's Corner 1 — List Comprehensions

## What is a List Comprehension?

A concise, Pythonic way to build a list in **one line** — faster and more readable than a for loop with `.append()`.

```python
# ❌ for-loop approach (verbose)
result = []
for x in iterable:
    result.append(f(x))

# ✅ list comprehension (Pythonic)
result = [f(x) for x in iterable]
```

---

## With Tuples (unpacking)

When each item in the iterable is a tuple, unpack all fields:

```python
ta_data = [('Peter', 'USA', 'CS212'),
           ('Sarah', 'France', 'CS101'),
           ('Gundega', 'Latvia', 'CS387')]

# ✅ readable unpacking
ta_facts = [name + " lives in " + country + " and TAs " + course
            for name, country, course in ta_data]
```

> Always unpack **all** fields in the tuple — leaving one out causes a `ValueError: too many values to unpack`.

---

## Filtering with `if`

Add an `if` clause to include only items that match a condition:

```python
# Only TAs outside the USA
remote = [name + " lives in " + country + " and TAs " + course
          for name, country, course in ta_data
          if country != 'USA']
```

---

## Pattern Summary

```python
[expression  for var in iterable]               # basic
[expression  for a, b, c in iterable]           # tuple unpacking
[expression  for a, b, c in iterable  if cond]  # with filter
```

**Exercise:** find all TAs teaching a 300-level course:
```python
ta_300 = [name for name, country, course in ta_data if course.startswith('3')]
# ['Gundega', 'Job']
```


### ✅ Exercise Solution — 300-level TAs

```python
ta_data = [['Peter',   'USA',     'CS262'],
           ['Andy',    'USA',     'CS212'],
           ['Sarah',   'England', 'CS101'],
           ['Gundega', 'Latvia',  'CS373'],
           ['Job',     'USA',     'CS387'],
           ['Sean',    'USA',     'CS253']]

ta_300 = [name + " is the TA for " + course
          for name, country, course in ta_data
          if course.find('CS3') != -1]

# → ['Gundega is the TA for CS373', 'Job is the TA for CS387']
```

**Key points:**
- `country` must be included in the unpacking even though it's not used — each entry has 3 elements and all must be referenced.
- `course.find('CS3') != -1` — `find()` returns `-1` when the substring is **not** found; `!= -1` means "CS3 was found" → 300-level course.
- `CS373` and `CS387` both contain `'CS3'` ✅ — `CS253` does not ❌.


---

# 🃏 Poker Rules — Quick Reference

Only one thing matters here: how 5-card hands are ranked (numbered 0–8, Python style):

| # | Hand | What it is |
|---|------|-----------|
| 8 | Straight Flush | 5 consecutive, same suit — `9H TH JH QH KH` |
| 7 | Four of a Kind | 4 same rank — `AS AH AD AC 6D` |
| 6 | Full House | 3 of one + 2 of another — `TH TC TS 2H 2D` |
| 5 | Flush | 5 same suit — `3S 6S JS QS KS` |
| 4 | Straight | 5 consecutive, any suit — `5S 6D 7C 8D 9S` |
| 3 | Three of a Kind | 3 same rank — `4H 4D 4C QS 2H` |
| 2 | Two Pair | 2 pairs — `JH JD 9H 9C 3S` |
| 1 | Pair | 2 same rank — `3H 3D 8C 5H 2S` |
| 0 | High Card | Nothing — `KH 9D 7S 4C 2H` |

Cards: `rank + suit` (e.g. `TH` = Ten of Hearts). Ties broken by card values.


---

## 🂡 HW1 — 7-Card Stud: `best_hand`

In 7-card stud you're dealt 7 cards but only play the best 5. The task: find that best 5-card hand.

**Key insight:** generate all possible 5-card combinations from 7 cards and pick the one with the highest `hand_rank`.

```python
import itertools

def best_hand(hand):
    "From a 7-card hand, return the best 5 card hand."
    return max(itertools.combinations(hand, 5), key=hand_rank)
```

**How it works — 3 steps:**

1. **`combinations(hand, 5)`** → generates all 21 possible 5-card subsets from the 7 cards
2. **`hand_rank`** → scores each combination with a comparable tuple e.g. `(8, 10)` for a straight flush
3. **`max`** → returns the combination that produced the highest tuple

```
7 cards → 21 combos → hand_rank each one → max → best 5-card hand
```

> Works for any number of input cards — not just 7.


---

## 🃏 HW1-2 — Jokers Wild: `best_wild_hand`

Two jokers act as **wild cards**, restricted by color:

| Joker | Can replace |
|-------|------------|
| `?B` (black) | Any spade `S` or club `C` |
| `?R` (red)   | Any heart `H` or diamond `D` |

**Strategy:**
1. For each card in the hand, define its possible values — normal cards map to themselves, jokers map to all 26 cards of their color.
2. Use `itertools.product` to generate every combination of substitutions.
3. For each substituted hand, find the best 5-card selection with `best_hand`.
4. Return the overall best.

```python
allranks   = '23456789TJQKA'
black_cards = [r+s for r in allranks for s in 'SC']  # 26 black cards
red_cards   = [r+s for r in allranks for s in 'HD']  # 26 red cards

def replacements(card):
    if card == '?B': return black_cards
    if card == '?R': return red_cards
    return [card]   # normal card → only itself

def best_wild_hand(hand):
    "Try all values for jokers in all 5-card selections."
    hands = set(best_hand(list(h))
                for h in itertools.product(*map(replacements, hand)))
    return max(hands, key=hand_rank)
```

**How it works:**
- `map(replacements, hand)` → for each card returns its list of options (1 option for normal cards, 26 for jokers)
- `itertools.product(...)` → generates every possible substituted hand (up to 26² = 676 if two jokers)
- `best_hand` → picks best 5 from each substituted hand
- `max(..., key=hand_rank)` → returns the overall best

> `set(...)` avoids evaluating duplicate hands when the same card appears multiple times.


---

# 🧩 Lesson 2 — Zebra Puzzle

## 📋 The Puzzle

**15 clues. 5 houses. Find: who drinks water? who owns the zebra?**

Each house is a different color; inhabitants differ in nationality, pet, drink, and cigarette brand.

| # | Clue |
|---|------|
| 1 | There are five houses |
| 2 | The Englishman lives in the red house |
| 3 | The Spaniard owns the dog |
| 4 | Coffee is drunk in the green house |
| 5 | The Ukrainian drinks tea |
| 6 | The green house is immediately to the right of the ivory house |
| 7 | The Old Gold smoker owns snails |
| 8 | Kools are smoked in the yellow house |
| 9 | Milk is drunk in the middle house |
| 10 | The Norwegian lives in the first house |
| 11 | The Chesterfields smoker lives next to the man with the fox |
| 12 | Kools are smoked next to the house with the horse |
| 13 | The Lucky Strike smoker drinks orange juice |
| 14 | The Japanese smokes Parliaments |
| 15 | The Norwegian lives next to the blue house |


## 📦 Concepts Inventory

| Concept | Description |
|---|---|
| **Houses** | 5 numbered positions (1–5) |
| **Properties** | Nationality, color, pet, drink, smoke |
| **Property group** | 5 mutually exclusive values assigned to the 5 houses |
| **Assignment** | Match each property value to exactly one house |
| **Location relations** | first, middle, next-to, immediately-right-of |

**Key design decision — do we need to name property groups?**

Yes. Properties **within a group are mutually exclusive** (if red → house 2, no other color can → house 2), but properties from **different groups are not** (orange juice can also → house 2). The program must track this.

Two valid approaches:
- Named groups: `nationality = [Englishman, Spaniard, ...]`
- Unnamed groups: manage grouping implicitly in the iteration

Ignoring groups entirely **does not work**.


## 🔧 Representing Assignments

Three valid implementation styles — all work:

```python
# Option A — house as set of properties
houses[1].add('red')

# Option B — house as object with fields
house1.color = 'red'

# Option C — property as variable (simplest ✅)
red = 1   # house number assigned to the property
```

**Norvig's choice: Option C** — assign a house number to each property variable. Simplest, and simple is enough until proven otherwise.


## 📐 Back-of-Envelope: Brute Force Feasibility

### How many combinations?

- Each property group has **5! = 120** possible orderings (permutations of 5 values across 5 houses)
- There are **5 property groups** → total = $5!^5$

$$5!^5 = 120^5 \approx 24{.}9 \text{ billion}$$

**Quick estimate:** round 120 → 100, then $100^5 = 10^{10}$ (10 billion). Round back up → ~20 billion. ✅

### Is it feasible?

| Scale | Verdict |
|-------|---------|
| Millions | ✅ Happy valley — fast |
| **Billions** | ❓ Uncertain — need to refine |
| Trillions | ❌ Infeasible — need a better algorithm |

At ~1 billion instructions/sec, **24.9 billion iterations ≈ ~1 hour** for a naïve brute force. Not fast enough — but can we prune?


## 🏗️ Brute Force Structure

```python
import itertools

houses = (1, 2, 3, 4, 5)
orderings = list(itertools.permutations(houses))  # 120 orderings

for (red, green, ivory, yellow, blue) in orderings:          # colors
  for (Englishman, Spaniard, Ukrainian, Norwegian, Japanese) in orderings:  # nationalities
    for (dog, snails, fox, horse, zebra) in orderings:       # pets
      for (OJ, tea, coffee, milk, water) in orderings:       # drinks
        for (OldGold, Kools, Chesterfields, LuckyStrike, Parliaments) in orderings:  # smokes
          if <all constraints satisfied>:
              solution found
```

- 5 nested loops × 120 orderings each = $120^5 \approx 25$ billion iterations
- Each property variable gets its house number directly (Option C representation)
- Constraints are checked **after** all 5 loops are nested — no early pruning

> 💡 `itertools.permutations(houses)` generates all 120 orderings of `(1,2,3,4,5)` — one per possible assignment of a property group to the 5 houses.

**Estimated runtime: ~1 hour** — in the uncertain zone. Can we improve by checking constraints earlier?


## ⏱️ Runtime Estimate (Back of Envelope)

$$5!^5 = 120^5 \approx 24.9 \text{ billion iterations}$$

| Instructions per iteration | Time at 2.4 GHz |
|---|---|
| 1 (impossible) | ~10 sec |
| 100 | ~16 min |
| **1,000 (realistic)** | **~160 min ≈ 2–3 hours** |

**Conclusion:** brute force with all constraints checked at the end → **~1–3 hours**. In the uncertain zone — but Norvig actually ran it to confirm. We need to prune constraints earlier to bring it into the "happy valley".


## 🔗 Encoding Constraints

With Option C (property = house number), each clue becomes a simple equality or helper call:

```python
# Clue 2: The Englishman lives in the red house
Englishman == red

# Clue 3: The Spaniard owns the dog
Spaniard == dog

# Clue 6: The green house is immediately to the right of the ivory house
imright(green, ivory)

# Clue 11: Chesterfields smoker lives next to the man with the fox
nextto(Chesterfields, fox)
```

### Helper functions

```python
def imright(h1, h2):
    "House h1 is immediately right of h2 if h1-h2 == 1."
    return h1 - h2 == 1

def nextto(h1, h2):
    "Two houses are next to each other if they differ by 1."
    return abs(h1 - h2) == 1
    # equivalently: imright(h1, h2) or imright(h2, h1)
```


## 🧩 Complete Solution — `zebra_puzzle`

Uses a **generator expression** instead of nested for-loops — same logic, cleaner structure.

```python
import itertools

def zebra_puzzle():
    "Return (WATER, ZEBRA) — house numbers for who drinks water and who owns the zebra."
    houses = [1, 2, 3, 4, 5]
    orderings = list(itertools.permutations(houses))  # 120 orderings
    return next(
        (WATER, ZEBRA)
        for (red, green, ivory, yellow, blue)                      in orderings
        for (Englishman, Spaniard, Ukrainian, Norwegian, Japanese) in orderings
        for (dog, snails, fox, horse, zebra)                       in orderings
        for (OJ, tea, coffee, milk, WATER)                         in orderings
        for (OldGold, Kools, Chesterfields, LuckyStrike, Parliaments) in orderings
        if Englishman == red                    # 2
        if Spaniard == dog                      # 3
        if coffee == green                      # 4
        if Ukrainian == tea                     # 5
        if imright(green, ivory)                # 6
        if OldGold == snails                    # 7
        if Kools == yellow                      # 8
        if milk == 3                            # 9 — middle house
        if Norwegian == 1                       # 10 — first house
        if nextto(Chesterfields, fox)           # 11
        if nextto(Kools, horse)                 # 12
        if LuckyStrike == OJ                    # 13
        if Japanese == Parliaments              # 14
        if nextto(Norwegian, blue)              # 15
    )
```

- `next(...)` returns the **first** yielded tuple from the generator — the solution.
- Each `if` is a **guard**: as soon as one fails, the inner loop skips that combination.
- All 15 constraints are encoded as simple equalities or helper calls.

> ⚠️ This version still checks all constraints **at the deepest nesting level** — runtime ~1–3 hours. See next section for the pruned version.


## 🔁 Generator Expressions vs. List Comprehensions

**List comprehension** → builds the entire list in memory:
```python
[s for r, s in cards]                            # all suits
[s for r, s in cards if r in ('J','Q','K')]      # only face card suits
```

**Generator expression** → same syntax but with `()` → produces values **on demand** (lazy):
```python
(s for r, s in cards)                            # generator — no list built yet
```

### How to read them — left to right (except the first term)

```
[ term  |  for a, b in iterable  |  if condition  |  for x in other  |  if ... ]
          ↑ read these first, left→right, as nested statements
                                                                         ↑ then evaluate term
```

The `term` at the front is evaluated **last**, once you reach the innermost level of nesting.

### `next()` + generator = get first match without computing all

```python
next(x for x in big_list if condition(x))   # stops at first match ✅
```

vs.

```python
[x for x in big_list if condition(x)][0]    # builds full list first ❌
```

> 💡 In `zebra_puzzle`, `next(...)` stops as soon as the first valid assignment is found — no wasted work after the solution is reached.


## ✂️ Pruning Zebra Puzzle — Move Constraints Up

**Problem:** all 15 constraints are checked at the deepest loop level → 25 billion combos.

**Fix:** move each constraint to the **earliest loop** where all its variables are already bound.

```python
return next(
    (WATER, ZEBRA)
    for (red, green, ivory, yellow, blue) in orderings
    if imright(green, ivory)                # ← moved up: only needs colors
    for (Englishman, Spaniard, Ukrainian, Norwegian, Japanese) in orderings
    if Englishman == red                    # ← moved up: needs colors + nationalities
    if Norwegian == 1
    if nextto(Norwegian, blue)
    for (dog, snails, fox, horse, ZEBRA) in orderings
    if Spaniard == dog
    for (OJ, tea, coffee, milk, WATER) in orderings
    if Ukrainian == tea
    if coffee == green
    if milk == 3
    for (OldGold, Kools, Chesterfields, LuckyStrike, Parliaments) in orderings
    if Kools == yellow
    if OldGold == snails
    if LuckyStrike == OJ
    if Japanese == Parliaments
    if nextto(Chesterfields, fox)
    if nextto(Kools, horse)
)
```

| Version | Time |
|---------|------|
| All constraints at end | ~1 hour |
| Constraints moved up | **< 1 ms** |

> 💡 Each early `if` prunes entire subtrees of combinations before they're explored. Same algorithm — drastically less work.


## ⏱️ `timedcall` — Timing Any Function

```python
import time

def timedcall(fn, *args):
    "Call fn(*args); return (elapsed_time, result)."
    t0 = time.perf_counter()
    result = fn(*args)
    t1 = time.perf_counter()
    return t1 - t0, result

# Usage
elapsed, answer = timedcall(zebra_puzzle)
# → (0.00028, (1, 5))
```

### `*args` — Pack & Unpack Arguments

| Location | Meaning |
|----------|---------|
| `def f(fn, *args)` | collect all extra args into a **tuple** called `args` |
| `fn(*args)` | **unpack** the tuple back into positional arguments |

```python
timedcall(max, [3,1,2])   # args = ([3,1,2],)  →  fn(*args) = max([3,1,2])
```

### Functions as First-Class Objects

Passing `fn` instead of `fn()` **delays execution** until the clock has started:

```python
timedcall(zebra_puzzle)   # ✅ passes the function — called inside, after t0
timedcall(zebra_puzzle()) # ❌ evaluates zebra_puzzle() first — clock misses it
```

> 💡 Functions are objects just like ints or lists — they can be passed, stored, and called later. This lets you control *when* execution happens.


## 🔁 `timedcalls` — Repeated Timing (Experimental Science)

One measurement isn't enough — repeat to reduce:
- **external events** (OS noise)
- **random variation**
- **measurement errors**

```python
def average(numbers):
    return sum(numbers) / float(len(numbers))

def timedcalls(n, fn, *args):
    """Call fn(*args) n times (int) or for up to n seconds (float).
    Returns (min, avg, max) of elapsed times."""
    if isinstance(n, int):
        times = [timedcall(fn, *args)[0] for _ in range(n)]
    else:
        times = []
        while sum(times) < n:
            times.append(timedcall(fn, *args)[0])
    return min(times), average(times), max(times)
```

| `n` type | Behavior |
|----------|----------|
| `int` (e.g. `10`) | repeat exactly 10 times |
| `float` (e.g. `10.0`) | repeat until 10 seconds have elapsed |

```python
timedcalls(5, zebra_puzzle)      # run 5 times
timedcalls(2.0, zebra_puzzle)    # run for ~2 seconds
# → (min, avg, max) in seconds
```

> 💡 `isinstance(n, int)` distinguishes `10` (int) from `10.0` (float) — Python treats them as different types even though the value looks the same.


## 🧩 Aspect-Oriented Programming

When designing a program, we juggle multiple concerns at once:

| Concern | What it addresses |
|---------|-------------------|
| **Correctness** | Does the program do what it should? |
| **Efficiency** | Does it run fast enough? |
| **Debugging** | Extra scaffolding to inspect/verify during dev |

The problem: all three get **interwoven** in the same code, creating a mess.

**Aspect-Oriented Programming (AOP)** is the idea of **separating these concerns** as much as possible — keep the core correctness logic distinct from efficiency and debugging code.

> 💡 We can't achieve perfect separation, but we should always strive to **minimize coupling** between these aspects.


## 🔢 Counting Iterations — Instrumenting with `c()`

When the solver is a **single generator expression**, you can't insert `count += 1` statements inside it. The solution: wrap `orderings` with a lightweight counting function `c()`.

```python
# Before: orderings used directly
for (red, green, ivory, yellow, blue) in orderings

# After: wrap with c() — minimal change, no logic altered
for (red, green, ivory, yellow, blue) in c(orderings)
```

`c()` is a **debugging tool** — short name is acceptable since it's temporary scaffolding.

### `instrument_fn` — Measure calls & items

```python
def instrument_fn(fn, *args):
    c.starts, c.items = 0, 0
    result = fn(*args)
    print(f'{c.starts} starts, {c.items} items')
    return result
```

| Metric | Meaning |
|--------|---------|
| `c.starts` | how many times the for loop started |
| `c.items` | how many items were yielded through |

> 💡 This gives you visibility into **how much work** the solver actually does, without changing the puzzle logic.


## ⚡ Generator Functions — `yield`

A **generator function** looks like a regular function but uses `yield` instead of `return`.

```python
def ints(start, end=None):
    i = start
    while i <= end or end is None:
        yield i
        i += 1
```

### How it works

- When called, it returns a **generator object** (not a list)
- Each call to `next()` runs until the next `yield`, returns that value, then **pauses**
- Resumes from the same point on the next `next()` call

```python
L = ints(0, 1_000_000)   # instant — no list created
next(L)   # → 0
next(L)   # → 1
# or: for i in L: ...
```

### Regular function vs Generator function

| | Regular function | Generator function |
|--|------------------|--------------------|
| Keyword | `return` | `yield` |
| Returns | a value (once) | a lazy sequence |
| Memory | stores all results | one item at a time |

### Infinite sequences

Pass `end=None` to get an **infinite generator**:

```python
ints(0)       # 0, 1, 2, 3, ... forever
```

The loop condition `while i <= end or end is None` keeps going when `end` is `None`.

> 💡 Generators are perfect for infinite or very large sequences — they only compute values **on demand**.


## 🔄 Exercise — `all_ints()`

Generate integers in the order: `0, +1, -1, +2, -2, +3, -3, ...`

```python
def all_ints():
    "Generate integers in the order 0, +1, -1, +2, -2, +3, -3, ..."
    yield 0
    for i in ints(1):        # infinite: 1, 2, 3, ...
        yield +i
        yield -i

# Alternative (explicit infinite loop):
def all_ints():
    yield 0
    i = 1
    while True:
        yield +i
        yield -i
        i += 1
```

> 💡 Both produce the same infinite sequence — `ints(1)` is more concise; `while True` is more explicit.


## 🔍 How `for` Loops Really Work

You might think `for x in items: print(x)` is just index-based iteration. The truth is deeper:

```python
# What Python actually does:
it = iter(items)          # 1. create an iterator
while True:
    try:
        x = next(it)      # 2. get next item
        print(x)          # 3. run the body
    except StopIteration:
        break             # 4. stop when exhausted
```

### Key concepts

| Term | Meaning |
|------|---------|
| **Iterable** | anything you can loop over: list, string, generator, ... |
| `iter(obj)` | creates an **iterator** from an iterable |
| `next(it)` | returns the next value; raises `StopIteration` when done |

- A **generator function** is already an iterator — `yield` handles `StopIteration` automatically
- Generators, lists, strings, and tuples all implement this same **iteration protocol**

> 💡 Because generators control *when* they produce values, we can count exactly how many items were consumed — this is how `c()` works.


## 🧮 `c()` — The Counting Generator

```python
def c(sequence):
    "Count the starts and items of a sequence."
    c.starts += 1
    for item in sequence:
        c.items += 1
        yield item
```

- `c.starts` increments each time the `for` loop begins a new `orderings` iteration
- `c.items` increments for every item yielded — i.e., items actually consumed
- Because `c` is a **generator**, it only counts items that are actually requested (early `if` filters stop it early)

```python
# Setup before calling the solver:
c.starts, c.items = 0, 0
instrument_fn(zebra_puzzle)
# → "25 starts, 2769 items"
```

> 💡 This separation is AOP in action: the counting concern lives in `c()`, not scattered across the puzzle logic.


## 📝 Zebra Puzzle — Lessons Learned

| Step | What we did |
|------|-------------|
| **Concept inventory** | Listed all entities and constraints before coding |
| **Simple first** | Implemented the straightforward (brute-force) solution |
| **Back-of-envelope estimate** | Calculated runtime before optimizing |
| **Refine** | Moved constraints up to prune the search space |
| **Build tools** | Built `timedcall`, `timedcalls`, `c()` as reusable debugging/profiling aids |
| **Separation of aspects** | Kept correctness, efficiency, and debugging code as separate as possible |

**Key takeaway:** By following a systematic process — concept inventory → simple implementation → estimate → refine — you can reliably arrive at an elegant, efficient solution without needing a clever flash of insight.

> 💡 The cleverness comes from the *process*, not luck.
